<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Cosmos3 Generator Audiovisual with TensorRT-LLM

This notebook calls already-running TensorRT-LLM VisualGen servers with direct `curl` requests from Python.

The examples are split into Cosmos3-Nano and Cosmos3-Super sections. Each section is self-contained, so you can run just one. The notebook covers TensorRT-LLM's stable server flow for text-to-image, text-to-video, and image-to-video generation.


## 1. Prerequisites

Use a running TensorRT-LLM server with Cosmos3 VisualGen support and set endpoint environment variables before the setup cell if you are not using the local default. Generation uses `/v1/videos/generations`; Cosmos3 text-to-image runs as a one-frame VisualGen video request with `num_frames=1`, `seconds=1`, and `fps=8`.

Generator requires the Guardrail. Request access to the gated [nvidia/Cosmos-1.0-Guardrail](https://huggingface.co/nvidia/Cosmos-1.0-Guardrail) HF repository before running these examples. TensorRT-LLM loads guardrails by default; to disable them, set `TRTLLM_DISABLE_COSMOS3_GUARDRAILS=1` before starting the server or set `use_guardrails` to `False` in the request `extra_params`.

```bash
export COSMOS3_TRTLLM_BASE_URL=http://localhost:8000
export COSMOS3_TRTLLM_NANO_BASE_URL=http://localhost:8000
export COSMOS3_TRTLLM_SUPER_BASE_URL=http://localhost:8000
export COSMOS3_TRTLLM_API_KEY=tensorrt_llm
```


## 2. Start the Server

Run the TensorRT-LLM VisualGen server before running the request cells. The config YAMLs below come from TensorRT-LLM's Cosmos3 support. To build a checkout from source, follow NVIDIA's [Build from Source](https://nvidia.github.io/TensorRT-LLM/installation/build-from-source.html) guide and then run the commands below from that checkout.

### Cosmos3-Nano

From the repository root, with TensorRT-LLM installed in the active environment:

```bash
export TRTLLM_ROOT="${TRTLLM_ROOT:-$PWD/TensorRT-LLM}"

trtllm-serve nvidia/Cosmos3-Nano   --visual_gen_args "$TRTLLM_ROOT/examples/visual_gen/configs/cosmos3-nano-1gpu.yaml"   --port 8000
```

### Cosmos3-Super

`Cosmos3-Super` uses the four-GPU config from TensorRT-LLM. The config sets `cfg_size=2`, `ulysses_size=2`, and `parallel_vae_size=4`, so launch exactly four processes.

```bash
export TRTLLM_ROOT="${TRTLLM_ROOT:-$PWD/TensorRT-LLM}"

torchrun --nproc_per_node=4 -m tensorrt_llm.commands.serve   nvidia/Cosmos3-Super   --visual_gen_args "$TRTLLM_ROOT/examples/visual_gen/configs/cosmos3-super-4gpu.yaml"   --port 8000
```

TensorRT-LLM exposes `/health` when the server is ready. This notebook sends Cosmos3 model-specific controls through `extra_params`, so use a TensorRT-LLM release that includes the Cosmos3 VisualGen API schema.


## 3. Configure Paths and Endpoints

This setup cell only configures repo/output paths and TensorRT-LLM endpoint settings.


In [ ]:
from pathlib import Path
import os


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "cookbooks").exists():
            return path
    return start


COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
COSMOS3_AUDIOVISUAL_ROOT = COSMOS_ROOT / "cookbooks" / "cosmos3" / "generator" / "audiovisual"
COSMOS3_AUDIOVISUAL_OUTPUT_ROOT = Path(
    os.environ.get("COSMOS3_AUDIOVISUAL_OUTPUT_ROOT", COSMOS3_AUDIOVISUAL_ROOT / "outputs" / "notebooks")
).resolve()
DEFAULT_TRTLLM_BASE_URL = os.environ.get("COSMOS3_TRTLLM_BASE_URL", "http://localhost:8000")
TRTLLM_ENDPOINTS = {
    "Cosmos3-Nano": os.environ.get("COSMOS3_TRTLLM_NANO_BASE_URL", DEFAULT_TRTLLM_BASE_URL),
    "Cosmos3-Super": os.environ.get("COSMOS3_TRTLLM_SUPER_BASE_URL", DEFAULT_TRTLLM_BASE_URL),
}

os.environ["COSMOS3_AUDIOVISUAL_OUTPUT_ROOT"] = str(COSMOS3_AUDIOVISUAL_OUTPUT_ROOT)
os.environ.setdefault("COSMOS3_TRTLLM_API_KEY", "tensorrt_llm")

print("COSMOS_ROOT:", COSMOS_ROOT)
print("COSMOS3_AUDIOVISUAL_OUTPUT_ROOT:", COSMOS3_AUDIOVISUAL_OUTPUT_ROOT)
for model, endpoint in TRTLLM_ENDPOINTS.items():
    print(f"{model} endpoint: {endpoint}")


## 4. Verify Endpoint Configuration


In [ ]:
from urllib.parse import urlparse


def api_root_url(base_url: str) -> str:
    normalized = base_url.rstrip("/")
    if not normalized.endswith("/v1"):
        normalized = f"{normalized}/v1"
    return normalized


def server_root_url(base_url: str) -> str:
    normalized = base_url.rstrip("/")
    if normalized.endswith("/v1"):
        normalized = normalized[:-3]
    return normalized.rstrip("/")


def health_url(base_url: str) -> str:
    return f"{server_root_url(base_url)}/health"


def video_api_url(base_url: str) -> str:
    return f"{api_root_url(base_url)}/videos/generations"


for model, base_url in TRTLLM_ENDPOINTS.items():
    parsed = urlparse(api_root_url(base_url))
    print(model)
    print("  api root:", api_root_url(base_url))
    print("  health:", health_url(base_url))
    print("  videos generations:", video_api_url(base_url))
    print("  scheme:", parsed.scheme)
    print("  host:", parsed.netloc)


## 5. Preview Available Inputs


In [ ]:
from pathlib import Path
import json
from IPython.display import Image, display

assets_dir = COSMOS3_AUDIOVISUAL_ROOT / "assets"
for prompt_dir in sorted((assets_dir / "prompts").iterdir()):
    if not prompt_dir.is_dir():
        continue
    print(f"{prompt_dir.relative_to(assets_dir)}:")
    for prompt_path in sorted(prompt_dir.glob("*.json")):
        data = json.loads(prompt_path.read_text())
        caption = (
            data.get("temporal_caption")
            or data.get("comprehensive_t2i_caption")
            or data.get("extra", {}).get("prompt", "")
        )
        print(f"  {prompt_path.name}: {caption[:180]}{'...' if len(caption) > 180 else ''}")
    print()

for image_dir in sorted((assets_dir / "images").iterdir()):
    if not image_dir.is_dir():
        continue
    print(f"{image_dir.relative_to(assets_dir)}:")
    for image_path in sorted(image_dir.iterdir()):
        if image_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}:
            print(f"  {image_path.name}")
            display(Image(filename=str(image_path), width=420))


## 6. Define Asset Sets, Payload Helpers, Request Helpers, and Viewer Helpers


In [ ]:
import base64
import html
import json
import os
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path
from IPython.display import HTML, Image, display


def api_root_url(base_url: str) -> str:
    normalized = base_url.rstrip("/")
    if not normalized.endswith("/v1"):
        normalized = f"{normalized}/v1"
    return normalized


def server_root_url(base_url: str) -> str:
    normalized = base_url.rstrip("/")
    if normalized.endswith("/v1"):
        normalized = normalized[:-3]
    return normalized.rstrip("/")


def health_url(base_url: str) -> str:
    return f"{server_root_url(base_url)}/health"


def video_api_url(base_url: str) -> str:
    return f"{api_root_url(base_url)}/videos/generations"


IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

FIXED_SAMPLING = {
    "num_steps": 35,
    "guidance": 6.0,
    "fps": 24,
    "num_frames": 189,
    "max_sequence_length": 2048,
    "resolution": "720",
    "aspect_ratio": "16,9",
    "seed": 0,
}

TRTLLM_EXTRA_PARAMS = {
    "use_resolution_template": False,
    "use_duration_template": False,
    "use_system_prompt": False,
    "use_guardrails": True,
}
# TensorRT-LLM Cosmos3 uses the stable video endpoint path here. Text-to-image
# generation is represented as a one-frame video request.
ASSET_SETS = {
    "t2i_nano": {
        "model": "Cosmos3-Nano",
        "mode": "text2image",
        "prompt": "assets/prompts/text2image/robot_draping.json",
    },
    "t2v_nano": {
        "model": "Cosmos3-Nano",
        "mode": "text2video",
        "prompt": "assets/prompts/text2video/robot_kitchen.json",
    },
    "i2v_nano": {
        "model": "Cosmos3-Nano",
        "mode": "image2video",
        "prompt": "assets/prompts/image2video/humanoid_robot.json",
        "image": "assets/images/image2video/humanoid_robot.jpg",
    },
    "t2i_super": {
        "model": "Cosmos3-Super",
        "mode": "text2image",
        "prompt": "assets/prompts/text2image/robot_draping.json",
    },
    "t2v_super": {
        "model": "Cosmos3-Super",
        "mode": "text2video",
        "prompt": "assets/prompts/text2video/robot_kitchen.json",
    },
    "i2v_super": {
        "model": "Cosmos3-Super",
        "mode": "image2video",
        "prompt": "assets/prompts/image2video/humanoid_robot.json",
        "image": "assets/images/image2video/humanoid_robot.jpg",
    },
}


def asset_path(relative_path: str) -> Path:
    path = COSMOS3_AUDIOVISUAL_ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(path)
    return path.resolve()


def compact_json_file(path: Path) -> str:
    return json.dumps(json.loads(path.read_text()), ensure_ascii=True, separators=(",", ":"))


def payload_dimensions(payload: dict) -> tuple[int, int]:
    if payload.get("resolution") == "720" and payload.get("aspect_ratio") == "16,9":
        return 720, 1280
    if payload.get("resolution") == "256" and payload.get("aspect_ratio") == "16,9":
        return 192, 320
    raise ValueError(f"Unsupported payload resolution/aspect ratio: {payload.get('resolution')} {payload.get('aspect_ratio')}")


def resolve_payload_path(payload_path: Path, value: str) -> Path:
    path = Path(value)
    if path.is_absolute():
        return path
    return (payload_path.parent / path).resolve()


def create_payload(use_case: str, *, backend: str = "trt_llm") -> tuple[Path, Path, str]:
    spec = ASSET_SETS[use_case]
    payload_dir = Path(os.environ["COSMOS3_AUDIOVISUAL_OUTPUT_ROOT"]) / backend / "payloads" / use_case
    output_dir = Path(os.environ["COSMOS3_AUDIOVISUAL_OUTPUT_ROOT"]) / backend / use_case
    payload_dir.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    prompt_path = asset_path(spec["prompt"])
    payload_path = payload_dir / f"{use_case}.json"
    payload = {
        "model_mode": spec["mode"],
        "name": use_case,
        "prompt": compact_json_file(prompt_path),
        "extra_params": dict(TRTLLM_EXTRA_PARAMS),
        **FIXED_SAMPLING,
    }
    if spec["mode"] == "text2image":
        payload["num_frames"] = 1
        payload["fps"] = 8
        payload["seconds"] = 1
    else:
        negative_prompt_path = asset_path(f"assets/negative_prompts/{spec['mode']}/neg_prompt.json")
        payload["negative_prompt"] = compact_json_file(negative_prompt_path)
    if spec["mode"] == "image2video":
        image_path = asset_path(spec["image"])
        payload["vision_path"] = os.path.relpath(image_path, payload_path.parent)

    payload_path.write_text(json.dumps(payload, indent=2) + "\n")

    os.environ[f"COSMOS3_{backend.upper()}_{use_case.upper()}_INPUT"] = str(payload_path)
    os.environ[f"COSMOS3_{backend.upper()}_{use_case.upper()}_OUTPUT"] = str(output_dir)

    print(f"model:   {spec['model']}")
    print(f"payload: {payload_path}")
    print(f"output:  {output_dir}")
    print(f"prompt:  {prompt_path.relative_to(COSMOS_ROOT)}")
    if spec["mode"] == "text2image":
        print("note:   TensorRT-LLM Cosmos3 text-to-image is served as a one-frame video response")
    if "vision_path" in payload:
        image_display_path = resolve_payload_path(payload_path, payload["vision_path"])
        print(f"image:   {image_display_path.relative_to(COSMOS_ROOT)}")
        display(Image(filename=str(image_display_path), width=420))
    preview_keys = ["model_mode", "name", "num_steps", "guidance", "fps", "num_frames", "max_sequence_length", "resolution", "aspect_ratio", "seed", "extra_params"]
    if "seconds" in payload:
        preview_keys.insert(5, "seconds")
    print(json.dumps({k: payload[k] for k in preview_keys}, indent=2))
    return payload_path, output_dir, spec["model"]


def check_trtllm_server(model: str, timeout_s: int = 1800, interval_s: int = 10) -> None:
    url = health_url(TRTLLM_ENDPOINTS[model])
    deadline = time.time() + timeout_s
    print(f"waiting for {model} server: {url}")
    last_error = None
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as response:
                if response.status == 200:
                    print(f"{model} server is ready")
                    return
                last_error = f"HTTP {response.status}"
        except (urllib.error.URLError, TimeoutError, OSError) as exc:
            last_error = str(exc)
        print(f"not ready yet: {last_error}")
        time.sleep(interval_s)
    raise TimeoutError(f"Timed out waiting for {model} server at {url}. Last error: {last_error}")


def build_trtllm_video_body(payload: dict) -> dict:
    height, width = payload_dimensions(payload)
    body = {
        "prompt": payload["prompt"],
        "size": f"{width}x{height}",
        "seconds": payload.get("seconds", payload["num_frames"] / payload["fps"]),
        "fps": payload["fps"],
        "num_frames": payload["num_frames"],
        "num_inference_steps": payload["num_steps"],
        "guidance_scale": payload["guidance"],
        "max_sequence_length": payload["max_sequence_length"],
        "seed": payload["seed"],
    }
    if payload.get("negative_prompt") is not None:
        body["negative_prompt"] = payload["negative_prompt"]
    body["extra_params"] = payload["extra_params"]
    return body


def _auth_headers() -> list[str]:
    api_key = os.environ.get("COSMOS3_TRTLLM_API_KEY", "")
    return ["-H", f"Authorization: Bearer {api_key}"] if api_key else []


def _video_extension_from_response(header_text: str, content_path: Path) -> str:
    lowered = header_text.lower()
    if "video/x-msvideo" in lowered or "video/avi" in lowered:
        return ".avi"
    head = content_path.read_bytes()[:16]
    if head.startswith(b"RIFF") and b"AVI" in head[:12]:
        return ".avi"
    if len(head) >= 12 and head[4:8] == b"ftyp":
        return ".mp4"
    return ".mp4"


def post_video(*, payload_path: Path, payload: dict, output_stem: Path, model: str) -> Path:
    url = video_api_url(TRTLLM_ENDPOINTS[model])
    tmp_path = Path(f"{output_stem}.tmp")
    header_path = Path(f"{output_stem}.headers.txt")
    error_path = Path(f"{output_stem}.error.txt")
    for path in [tmp_path, header_path, error_path]:
        if path.exists():
            path.unlink()

    body = build_trtllm_video_body(payload)
    cmd = [
        "curl",
        "-sS",
        "--fail-with-body",
        "-X",
        "POST",
        url,
        "-D",
        str(header_path),
        "-H",
        "Accept: video/mp4, video/x-msvideo, application/octet-stream",
    ]
    cmd += _auth_headers()

    if payload["model_mode"] == "image2video":
        form_body = dict(body)
        if "extra_params" in form_body:
            form_body["extra_params"] = json.dumps(form_body["extra_params"], separators=(",", ":"))
        for key, value in form_body.items():
            cmd += ["--form-string", f"{key}={value}"]
        image_path = resolve_payload_path(payload_path, payload["vision_path"])
        cmd += ["-F", f"input_reference=@{image_path}"]
    else:
        cmd += ["-H", "Content-Type: application/json"]
        cmd += ["-d", json.dumps(body, separators=(",", ":"))]

    cmd += ["-o", str(tmp_path)]
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if result.returncode != 0:
        error_path.write_text((result.stdout or "") + (result.stderr or ""))
        raise RuntimeError(f"TensorRT-LLM request failed with exit code {result.returncode}; see {error_path}")

    header_text = header_path.read_text() if header_path.exists() else ""
    ext = _video_extension_from_response(header_text, tmp_path)
    output_path = output_stem.with_suffix(ext)
    if output_path.exists():
        output_path.unlink()
    tmp_path.replace(output_path)
    return output_path


def run_trtllm_payload(payload_path: Path, output_dir: str | Path, *, model: str) -> Path:
    payload_path = Path(payload_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    payload = json.loads(payload_path.read_text())
    output_stem = output_dir / payload["name"]
    endpoint = video_api_url(TRTLLM_ENDPOINTS[model])
    print("endpoint:", endpoint)
    print("payload:", payload_path)
    print("output stem:", output_stem)
    if payload["model_mode"] == "image2video":
        print("input image:", resolve_payload_path(payload_path, payload["vision_path"]))
    t0 = time.time()
    output_path = post_video(payload_path=payload_path, payload=payload, output_stem=output_stem, model=model)
    print(f"wrote {output_path} in {time.time() - t0:.1f}s")
    return output_path


def display_video(path: Path, *, width: int = 720) -> None:
    suffix = path.suffix.lower()
    media_type = "video/x-msvideo" if suffix == ".avi" else "video/mp4"
    data = base64.b64encode(path.read_bytes()).decode("ascii")
    label = html.escape(str(path))
    markup = f"""
<video controls playsinline preload="metadata" width="{width}" style="max-width: 100%; background: #000;">
  <source src="data:{media_type};base64,{data}" type="{media_type}">
</video>
<div style="font-family: monospace; font-size: 12px; margin-top: 4px;">{label}</div>
"""
    display(HTML(markup))


def view_run(output_dir: str | Path) -> None:
    output_dir = Path(output_dir)
    videos = [
        path
        for path in sorted(output_dir.rglob("*"))
        if path.suffix.lower() in {".mp4", ".avi"}
        and not path.name.endswith(("_preview.mp4", "_browser.mp4"))
    ]
    if not videos:
        print(f"No generated videos found under {output_dir}")
        return
    for src in videos:
        print(f"source: {src} ({src.stat().st_size // 1024} KB)")
        display_video(src)


Run each use case top-to-bottom: create the JSON payload, wait for the matching server, run inference, then view the generated media. The Cosmos3-Nano and Cosmos3-Super sections are independent, so you can run just one.


# Cosmos3-Nano Examples

Use cases for the `Cosmos3-Nano` model. This section is self-contained; you can run it without the Cosmos3-Super section below.


## Nano: Text to Image

Nano text-to-image generation using a structured JSON prompt. TensorRT-LLM Cosmos3 returns this as a one-frame video from the video generation endpoint; the request uses `num_frames=1`, `seconds=1`, and `fps=8`.


In [ ]:
t2i_nano_payload, t2i_nano_output, t2i_nano_model = create_payload("t2i_nano")


### Run


In [ ]:
check_trtllm_server(t2i_nano_model)
run_trtllm_payload(t2i_nano_payload, t2i_nano_output, model=t2i_nano_model)


### View Results


In [ ]:
view_run(t2i_nano_output)


## Nano: Text to Video

Nano text-to-video generation using a structured JSON prompt.


In [ ]:
t2v_nano_payload, t2v_nano_output, t2v_nano_model = create_payload("t2v_nano")


### Run


In [ ]:
check_trtllm_server(t2v_nano_model)
run_trtllm_payload(t2v_nano_payload, t2v_nano_output, model=t2v_nano_model)


### View Results


In [ ]:
view_run(t2v_nano_output)


## Nano: Image to Video

Nano image-to-video generation using its paired image asset.


In [ ]:
i2v_nano_payload, i2v_nano_output, i2v_nano_model = create_payload("i2v_nano")


### Run


In [ ]:
check_trtllm_server(i2v_nano_model)
run_trtllm_payload(i2v_nano_payload, i2v_nano_output, model=i2v_nano_model)


### View Results


In [ ]:
view_run(i2v_nano_output)


# Cosmos3-Super Examples

The same use cases for the larger `Cosmos3-Super` model. This section is self-contained; you can run it without the Cosmos3-Nano section above.


## Super: Text to Image

Super text-to-image generation using the same structured JSON prompt. TensorRT-LLM Cosmos3 returns this as a one-frame video from the video generation endpoint; the request uses `num_frames=1`, `seconds=1`, and `fps=8`.


In [ ]:
t2i_super_payload, t2i_super_output, t2i_super_model = create_payload("t2i_super")


### Run


In [ ]:
check_trtllm_server(t2i_super_model)
run_trtllm_payload(t2i_super_payload, t2i_super_output, model=t2i_super_model)


### View Results


In [ ]:
view_run(t2i_super_output)


## Super: Text to Video

Super text-to-video generation using the same structured JSON prompt.


In [ ]:
t2v_super_payload, t2v_super_output, t2v_super_model = create_payload("t2v_super")


### Run


In [ ]:
check_trtllm_server(t2v_super_model)
run_trtllm_payload(t2v_super_payload, t2v_super_output, model=t2v_super_model)


### View Results


In [ ]:
view_run(t2v_super_output)


## Super: Image to Video

Super image-to-video generation using its paired image asset.


In [ ]:
i2v_super_payload, i2v_super_output, i2v_super_model = create_payload("i2v_super")


### Run


In [ ]:
check_trtllm_server(i2v_super_model)
run_trtllm_payload(i2v_super_payload, i2v_super_output, model=i2v_super_model)


### View Results


In [ ]:
view_run(i2v_super_output)
